In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Galaxy Images : validation dataset & generative models

In [ ]:
n=16  # number of images to display per set
ntrain=100_000 # number of training samples used for each model

In [ ]:
#training data set
data_original = torch.load('../datasets/sdss_train_no_repeats_64x64.pt')  # shape [N,1,H,W]
test_idx=100000
data_original = data_original[test_idx:test_idx+n]   # n-images not used by any models
data_original = data_original.numpy()

In [ ]:
#diffusion model trained with 100,000 images
rootdir = "../diffusion_model_sdss/results/img_align_sdss_64x64/"
diffusion_A_B_samples = torch.load(rootdir+ "UNet_many_samples_"+str(ntrain)+"_10000.pt",
                                   map_location=torch.device('cpu'))
diffusion_samples = diffusion_A_B_samples['A']
diffusion_samples = diffusion_samples[:n].numpy()

In [ ]:
#Glow model trained with 100,000 images
rootdir = '../glow-pytorch-sdss/archive_2nd_model_cured_glow_sdss/'
glow_A_samples_1 = torch.load(rootdir + "glow_samples_"+str(ntrain)+"_2ndmod_A_4000_T1.0_1.pt",
                            map_location=torch.device('cpu'))
glow_samples = glow_A_samples_1['A'][:n].numpy()

In [ ]:
#Gan model trained with 100,000 images
rootdir = "../lightweight-gan/results/"
#nb below 147 is the last epoch used for ntrain=100000
epoch = 147 if int(ntrain) == 100_000 else 150
gan_samples = np.load(rootdir+"train_"+str(ntrain)+"_noaugatall_A-generated-"+str(epoch)+"/gan-samples_10000.npy")
gan_samples = gan_samples[:n]
gan_samples= np.expand_dims(gan_samples, axis=(1,))

In [ ]:
samples_type = [data_original,gan_samples,glow_samples,diffusion_samples]

In [ ]:
samples_type_name=["Original","GAN","Glow","Diff."]

In [ ]:
n_rows = len(samples_type)
n_columns = n
im_size = 1
fig, axs = plt.subplots(n_rows, n_columns, figsize = (im_size * n_columns , im_size * n_rows))
vmin = None
vmax = None
axs = axs.ravel()
i=0
for idx,dataset in enumerate(samples_type):
    axs[i].set_ylabel(samples_type_name[idx],fontsize=15)
    for j in range(n_columns):
        axs[i].imshow(dataset[j,0], "gray",vmin=vmin, vmax = vmax)
        i=i+1



for i in range(len(axs)):
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=0)

plt.savefig("fig-model_sample_images.pdf",bbox_inches='tight', pad_inches=0.1)
plt.show()